<a href="https://colab.research.google.com/github/harshkumar0901/Colab/blob/main/FastAPI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
import os

# Check if the model is already present (e.g., if you uploaded it manually or from Drive)
model_path = 'cifar10_cnn_model.keras'

if not os.path.exists(model_path):
    print(f"'{model_path}' not found. Please upload your model file:")
    uploaded = files.upload()
    if model_path not in uploaded:
        print(f"Warning: '{model_path}' was not uploaded. Please ensure you upload the correct file.")
    else:
        print(f"Successfully uploaded {model_path}")
else:
    print(f"'{model_path}' already exists. Skipping upload.")


'cifar10_cnn_model.keras' not found. Please upload your model file:


Saving cifar10_cnn_model.keras to cifar10_cnn_model.keras
Successfully uploaded cifar10_cnn_model.keras


In [2]:
# Cell 3: FastAPI Application Code

import nest_asyncio
nest_asyncio.apply()

from fastapi import FastAPI, UploadFile, File, HTTPException
from pydantic import BaseModel
import uvicorn
from threading import Thread
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import numpy as np
from PIL import Image
import io
import os
import asyncio # Import asyncio

# Define image dimensions (must match your model's input)
IMG_WIDTH = 32
IMG_HEIGHT = 32

# Define the path to your saved model
MODEL_PATH = 'cifar10_cnn_model.keras'

# Define the class labels in the correct order
# This order should match the order in which your LabelEncoder mapped them.
CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

# Initialize FastAPI app
app = FastAPI(
    title="CIFAR-10 Image Classifier API",
    description="A simple API to classify CIFAR-10 images using a pre-trained CNN model.",
    version="1.0.0",
)

# Global variable for the model (will be loaded once)
model = None

# Pydantic model for prediction response
class PredictionResponse(BaseModel):
    filename: str
    predicted_class: str
    confidence: float
    all_confidences: dict

# Function to load the model
@app.on_event("startup")
async def load_trained_model():
    global model
    try:
        model = load_model(MODEL_PATH)
        print(f"Model '{MODEL_PATH}' loaded successfully!")
    except Exception as e:
        raise RuntimeError(f"Error loading model from {MODEL_PATH}: {e}")

# Function to preprocess the image
def preprocess_image(img_file: UploadFile):
    try:
        # Read image file
        img_bytes = img_file.file.read()
        img = Image.open(io.BytesIO(img_bytes))

        # Convert to RGB if not already (handles RGBA or grayscale)
        if img.mode != 'RGB':
            img = img.convert('RGB')

        # Resize image
        img = img.resize((IMG_WIDTH, IMG_HEIGHT))

        # Convert to numpy array and normalize
        img_array = image.img_to_array(img)
        img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension
        img_array /= 255.0  # Normalize pixel values to [0, 1]
        return img_array
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"Image preprocessing failed: {e}")

# Root endpoint
@app.get("/", summary="Root endpoint")
async def read_root():
    return {"message": "Welcome to the CIFAR-10 Image Classifier API! Visit /docs for API documentation."}

# Prediction endpoint
@app.post("/predict/", response_model=PredictionResponse, summary="Classify an uploaded image")
async def predict_image(file: UploadFile = File(..., description="Upload an image file (PNG, JPG)")):
    if model is None:
        raise HTTPException(status_code=503, detail="Model not loaded. Please try again later.")

    processed_image = preprocess_image(file)

    # Make prediction
    predictions = model.predict(processed_image)[0]
    predicted_class_index = np.argmax(predictions)
    confidence = float(predictions[predicted_class_index])
    predicted_class_name = CLASS_NAMES[predicted_class_index]

    # Get all confidences as a dictionary
    all_confidences = {CLASS_NAMES[i]: float(predictions[i]) for i in range(len(CLASS_NAMES))}

    return PredictionResponse(
        filename=file.filename,
        predicted_class=predicted_class_name,
        confidence=confidence,
        all_confidences=all_confidences
    )

# Function to run Uvicorn in a separate thread with its own event loop
def run_uvicorn_in_thread():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio") # Explicitly specify asyncio loop
    server = uvicorn.Server(config)
    loop.run_until_complete(server.serve()) # Run until the server stops

# Start Uvicorn server in a new thread
print("Starting Uvicorn server...")
thread = Thread(target=run_uvicorn_in_thread)
thread.daemon = True # Allow main program to exit even if thread is still running
thread.start()

print("FastAPI application is running locally on port 8000. To access it externally, you'll need a tunneling service like ngrok.")
print("Open http://localhost:8000/docs for API documentation.")

Starting Uvicorn server...
FastAPI application is running locally on port 8000. To access it externally, you'll need a tunneling service like ngrok.
Open http://localhost:8000/docs for API documentation.


/tmp/ipykernel_3631/1147468928.py:47: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
        
  @app.on_event("startup")


In [4]:
!pip install pyngrok
from pyngrok import ngrok
NGROK_AUTH_TOKEN = "3DyZfvWiXxyZKyikURKfkLYo1sf_bdUKjmkkGqBSSH2kXDdA"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

In [5]:
public_url = ngrok.connect(8000)
print(f"Ngrok Public URL: {public_url}")
print(f"Open this URL in your browser to access your API and its documentation (e.g., {public_url}/docs)")

Ngrok Public URL: NgrokTunnel: "https://hemlock-crown-dealing.ngrok-free.dev" -> "http://localhost:8000"
Open this URL in your browser to access your API and its documentation (e.g., NgrokTunnel: "https://hemlock-crown-dealing.ngrok-free.dev" -> "http://localhost:8000"/docs)
